# 🍱 식단 데이터 준비

GROUP1~5 CSV 파일을 합쳐서 `food_nutrition.csv` 하나로 만드는 노트북

**실행 순서:** 위에서부터 순서대로 셀을 실행 (Shift+Enter)

## 1단계: 파일 불러오기 및 합치기

In [1]:
import pandas as pd
import glob

# ── GROUP1~5 파일 자동 탐색 ──────────────────────────────────────
# 이 노트북(prepare_data.ipynb)과 같은 폴더에
# FOOD-DATA-GROUP1.csv ~ FOOD-DATA-GROUP5.csv 를 넣어주세요
files = sorted(glob.glob("FOOD-DATA-GROUP*.csv"))

print(f"발견된 파일 {len(files)}개:")
for f in files:
    print(f"  {f}")

발견된 파일 5개:
  FOOD-DATA-GROUP1.csv
  FOOD-DATA-GROUP2.csv
  FOOD-DATA-GROUP3.csv
  FOOD-DATA-GROUP4.csv
  FOOD-DATA-GROUP5.csv


In [2]:
# ── 파일 합치기 ───────────────────────────────────────────────────
dfs = []
for f in files:
    df_tmp = pd.read_csv(f)
    print(f"{f}: {len(df_tmp)}행")
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)
print(f"\n합친 후 총 행 수: {len(df)}")

FOOD-DATA-GROUP1.csv: 551행
FOOD-DATA-GROUP2.csv: 319행
FOOD-DATA-GROUP3.csv: 571행
FOOD-DATA-GROUP4.csv: 232행
FOOD-DATA-GROUP5.csv: 722행

합친 후 총 행 수: 2395


## 2단계: 데이터 확인

In [3]:
# 컬럼 목록 확인
print("컬럼 목록:")
print(df.columns.tolist())

컬럼 목록:
['Unnamed: 0.1', 'Unnamed: 0', 'food', 'Caloric Value', 'Fat', 'Saturated Fats', 'Monounsaturated Fats', 'Polyunsaturated Fats', 'Carbohydrates', 'Sugars', 'Protein', 'Dietary Fiber', 'Cholesterol', 'Sodium', 'Water', 'Vitamin A', 'Vitamin B1', 'Vitamin B11', 'Vitamin B12', 'Vitamin B2', 'Vitamin B3', 'Vitamin B5', 'Vitamin B6', 'Vitamin C', 'Vitamin D', 'Vitamin E', 'Vitamin K', 'Calcium', 'Copper', 'Iron', 'Magnesium', 'Manganese', 'Phosphorus', 'Potassium', 'Selenium', 'Zinc', 'Nutrition Density']


In [4]:
# 첫 5행 미리보기
df.head()

,Unnamed: 0.1,Unnamed: 0,food,Caloric Value,Fat,Saturated Fats,Monounsaturated Fats,Polyunsaturated Fats,Carbohydrates,Sugars,...,Calcium,Copper,Iron,Magnesium,Manganese,Phosphorus,Potassium,Selenium,Zinc,Nutrition Density
0,0,0,cream cheese,51,5.0,2.9,1.3,0.200,0.8,0.500,...,0.008,14.100,0.082,0.027,1.300,0.091,15.5,19.100,0.039,7.070
1,1,1,neufchatel cheese,215,19.4,10.9,4.9,0.800,3.1,2.700,...,99.500,0.034,0.100,8.500,0.088,117.300,129.2,0.054,0.700,130.100
2,2,2,requeijao cremoso light catupiry,49,3.6,2.3,0.9,0.000,0.9,3.400,...,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,5.400
3,3,3,ricotta cheese,30,2.0,1.3,0.5,0.002,1.5,0.091,...,0.097,41.200,0.097,0.096,4.000,0.024,30.8,43.800,0.035,5.196
4,4,4,cream cheese low fat,30,2.3,1.4,0.6,0.042,1.2,0.900,...,22.200,0.072,0.008,1.200,0.098,22.800,37.1,0.034,0.053,27.007


In [5]:
# 기본 통계
df[["Caloric Value", "Fat", "Carbohydrates", "Protein"]].describe()

,Caloric Value,Fat,Carbohydrates,Protein
count,2395.000000,2395.000000,2395.000000,2395.000000
mean,223.769520,10.176276,18.589021,13.400777
std,384.728244,29.008915,29.406134,32.294246
min,0.000000,0.000000,0.000000,0.000000
25%,44.500000,0.300000,0.500000,0.800000
50%,117.000000,2.100000,6.800000,3.500000
75%,258.000000,9.400000,25.050000,13.300000
max,6077.000000,550.700000,390.200000,560.300000


## 3단계: 정리 및 저장

In [6]:
# ── 불필요한 인덱스 컬럼 제거 ────────────────────────────────────
df = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], errors="ignore")

# ── 결측치 확인 ───────────────────────────────────────────────────
print("결측치 개수:")
print(df[["food", "Caloric Value", "Fat", "Carbohydrates", "Protein"]].isnull().sum())

결측치 개수:
food             0
Caloric Value    0
Fat              0
Carbohydrates    0
Protein          0
dtype: int64


In [7]:
# ── 핵심 컬럼 결측치만 제거 ───────────────────────────────────────
before = len(df)
df = df.dropna(subset=["food", "Caloric Value", "Fat", "Carbohydrates", "Protein"])
print(f"결측치 제거: {before} → {len(df)}행")

# ── 중복 음식 제거 ────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset=["food"]).reset_index(drop=True)
print(f"중복 제거:   {before} → {len(df)}행")

print(f"\n최종 음식 수: {len(df)}개")

결측치 제거: 2395 → 2395행
중복 제거:   2395 → 2395행

최종 음식 수: 2395개


In [8]:
# ── food_nutrition.csv 로 저장 ────────────────────────────────────
# 이 파일을 environment.py가 읽어요
df.to_csv("food_nutrition.csv", index=False)
print("저장 완료: data/food_nutrition.csv")
print(f"최종 컬럼: {df.columns.tolist()}")

저장 완료: data/food_nutrition.csv
최종 컬럼: ['food', 'Caloric Value', 'Fat', 'Saturated Fats', 'Monounsaturated Fats', 'Polyunsaturated Fats', 'Carbohydrates', 'Sugars', 'Protein', 'Dietary Fiber', 'Cholesterol', 'Sodium', 'Water', 'Vitamin A', 'Vitamin B1', 'Vitamin B11', 'Vitamin B12', 'Vitamin B2', 'Vitamin B3', 'Vitamin B5', 'Vitamin B6', 'Vitamin C', 'Vitamin D', 'Vitamin E', 'Vitamin K', 'Calcium', 'Copper', 'Iron', 'Magnesium', 'Manganese', 'Phosphorus', 'Potassium', 'Selenium', 'Zinc', 'Nutrition Density']


## 4단계: environment.py 연동 테스트

In [9]:
import sys
sys.path.append("..")   # diet_rl/ 폴더를 경로에 추가

from environment import DietEnv

env = DietEnv("food_nutrition.csv")
print(f"음식 수 (action 가짓수): {env.n_foods}")
print(f"state 크기: {env.state_size}")

[Environment] 음식 데이터 로드 완료: 2395개 음식
[Environment] action space 제한: 500개 음식으로 랜덤 샘플링
음식 수 (action 가짓수): 500
state 크기: 6


In [10]:
# 하루 시뮬레이션 테스트
import numpy as np

state = env.reset()
print(f"초기 state: {state}")

total_reward = 0
while True:
    action = np.random.randint(env.n_foods)   # 임시로 랜덤 선택
    state, reward, done = env.step(action)
    total_reward += reward
    if done:
        break

env.render()
print(f"\n총 reward: {total_reward:.3f}")

초기 state: [1. 1. 1. 1. 0. 0.]

=== 주간 식단 (21/21 끼니) ===
    월요일: 아침: beef salami cooked
    월요일: 점심: bean with bacon soup
    월요일: 저녁: navy beans canned
    화요일: 아침: tripe soup
    화요일: 점심: dock
    화요일: 저녁: corn flour yellow
    수요일: 아침: nance
    수요일: 점심: dock
    수요일: 저녁: biscuit large mcdonalds
    목요일: 아침: herring fish oil
    목요일: 점심: hotcakes with syrup mcdonalds
    목요일: 저녁: duck meat cooked
    금요일: 아침: egg bagel
    금요일: 점심: witloof chicory
    금요일: 저녁: pork blade chops cooked
    토요일: 아침: vienna bread
    토요일: 점심: tuna salad
    토요일: 저녁: melba toast crackers
    일요일: 아침: macadamia nuts raw
    일요일: 점심: oriental rice cracker mix
    일요일: 저녁: chicken dumplings soup

   [남은 영양소 (일주일 기준)]
    calories       :   9960.0 남음 (달성률 28.9%)
    carbohydrates  :   1305.3 남음 (달성률 25.4%)
    protein        :    224.1 남음 (달성률 46.6%)
    fat            :    287.4 남음 (달성률 36.8%)

총 reward: -53.975
